# AdjointRWM A100 Production Pilot

This notebook trains a real-data, action-conditioned recursive world model on the official DROID `droid_100` RLDS sample, then fits and evaluates:

- a causal amortized co-state estimator;
- an information-equivalent direct marginal-gain critic;
- recursive refinement candidates under explicit rate/compute costs;
- a selective invocation gate that chooses between adjoint and critic paths.

## Scientific gates

1. No synthetic, random, hash-only, or unrelated-data fallback.
2. Split by complete episode before temporal windows are generated.
3. Train against actual future states and frozen future visual features.
4. Do not open adjoint training unless held-out dynamics beat persistence.
5. Do not report task success without a simulator or physical-robot evaluation.
6. Save configs, source/data manifests, logs, tests, benchmarks, artifacts, plots, and atomic checkpoints to an immutable Google Drive run directory.
7. Treat the co-state as objective sensitivity, not as action reconstruction or a generic auxiliary feature.

In [ ]:
# Install dependencies not guaranteed by the Colab image.
%pip -q install --upgrade "tensorflow-datasets>=4.9" rlds pyarrow tensorboard nbformat

In [ ]:
# ================================================================
# 1. Runtime, configuration, reproducibility, and Drive persistence
# ================================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
from dataclasses import dataclass, asdict
import copy
import datetime
import hashlib
import json
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import textwrap
import time
import traceback
import zlib

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler, default_collate
from torch.utils.tensorboard import SummaryWriter

assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'

DEVICE = torch.device('cuda')
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3

print('GPU:', GPU_NAME)
print(f'VRAM: {GPU_VRAM_GIB:.2f} GiB')
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)

if not any(name in GPU_NAME for name in ('A100', 'H100', 'L4')):
    raise RuntimeError(f'Unsupported training hardware profile: {GPU_NAME}')

@dataclass(frozen=True)
class Config:
    seed: int = 20260928

    # Official 100-trajectory DROID prototyping sample.
    dataset_name: str = 'droid_100'
    dataset_data_dir: str = 'gs://gresearch/robotics'
    max_episodes: int = 100
    frame_stride: int = 2

    # Temporal task.
    context_len: int = 8
    horizon: int = 4
    window_stride: int = 2

    # Frozen visual target encoder.
    visual_embedding_dim_per_camera: int = 512
    image_size: int = 128
    embedding_batch_size: int = 64

    # Model.
    d_model: int = 512
    transformer_layers: int = 6
    transformer_heads: int = 8
    transformer_ff: int = 2048
    dropout: float = 0.10
    num_refinement_candidates: int = 4

    # Training.
    dynamics_steps: int = 1500
    allocator_steps: int = 750
    eval_interval: int = 100
    checkpoint_interval: int = 250
    learning_rate: float = 3.0e-4
    allocator_learning_rate: float = 1.0e-4
    weight_decay: float = 0.05
    warmup_steps: int = 100
    gradient_clip: float = 1.0

    # Explicit rate/compute trade-off.
    rate_beta: float = 0.002
    candidate_cost_weight: float = 0.002
    candidate_costs: tuple = (1.0, 1.0, 1.5, 2.0)

    # Measured batch selection.
    microbatch_candidates: tuple = (8, 16, 32, 64)

    # Promotion criterion.
    baseline_improvement_margin: float = 0.02

CFG = Config()

RUN_ID = 'droid100_adjoint_' + datetime.datetime.now(
    datetime.UTC
).strftime('%Y%m%dT%H%M%SZ')

DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdjointRWM_Production')
RUN_DIR = DRIVE_ROOT / 'runs' / RUN_ID
LOCAL_CACHE = Path('/content/adjoint_rwm_cache') / RUN_ID
LOCAL_WORK = Path('/content/adjoint_rwm_work') / RUN_ID

RUN_PATHS = {
    'config': RUN_DIR / 'config',
    'checkpoints': RUN_DIR / 'checkpoints',
    'logs': RUN_DIR / 'logs',
    'tests': RUN_DIR / 'tests',
    'benchmarks': RUN_DIR / 'benchmarks',
    'artifacts': RUN_DIR / 'artifacts',
    'plots': RUN_DIR / 'artifacts' / 'plots',
    'samples': RUN_DIR / 'samples',
    'reports': RUN_DIR / 'reports',
}
for path in [RUN_DIR, LOCAL_CACHE, LOCAL_WORK, *RUN_PATHS.values()]:
    path.mkdir(parents=True, exist_ok=True)

LOCAL_EPISODES = LOCAL_CACHE / 'episodes'
LOCAL_EPISODES.mkdir(parents=True, exist_ok=True)


def stable_json_bytes(value):
    return json.dumps(
        value, sort_keys=True, separators=(',', ':'), default=str
    ).encode('utf-8')


def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()


def sha256_file(path, chunk_size=4 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(chunk_size), b''):
            digest.update(block)
    return digest.hexdigest()


def atomic_write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(text, encoding='utf-8')
    os.replace(tmp, path)


def atomic_write_json(path, value):
    atomic_write_text(
        path, json.dumps(value, indent=2, sort_keys=True, default=str)
    )


def atomic_copy(local_path, final_path):
    local_path = Path(local_path)
    final_path = Path(final_path)
    final_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = final_path.with_suffix(final_path.suffix + '.tmp')
    shutil.copy2(local_path, tmp)
    os.replace(tmp, final_path)
    digest = sha256_file(final_path)
    atomic_write_text(
        final_path.with_suffix(final_path.suffix + '.sha256'), digest + '\n'
    )
    return digest

METRICS_LOCAL = LOCAL_WORK / 'metrics.jsonl'
WARNINGS_LOCAL = LOCAL_WORK / 'warnings.jsonl'
TB_LOCAL = LOCAL_WORK / 'tensorboard'
TB_LOCAL.mkdir(parents=True, exist_ok=True)


def append_jsonl(path, value):
    with open(path, 'a', encoding='utf-8') as handle:
        handle.write(json.dumps(value, sort_keys=True, default=str) + '\n')


def log_metric(stage, step, **values):
    record = {
        'timestamp_utc': datetime.datetime.now(datetime.UTC).isoformat(),
        'stage': stage,
        'step': int(step),
        **values,
    }
    append_jsonl(METRICS_LOCAL, record)
    return record


def log_warning(code, message, **context):
    append_jsonl(
        WARNINGS_LOCAL,
        {
            'timestamp_utc': datetime.datetime.now(datetime.UTC).isoformat(),
            'code': code,
            'message': message,
            'context': context,
        },
    )


def sync_runtime_logs():
    if METRICS_LOCAL.exists():
        atomic_copy(METRICS_LOCAL, RUN_PATHS['logs'] / 'metrics.jsonl')
    if WARNINGS_LOCAL.exists():
        atomic_copy(WARNINGS_LOCAL, RUN_PATHS['logs'] / 'warnings.jsonl')
    if TB_LOCAL.exists():
        shutil.copytree(
            TB_LOCAL, RUN_PATHS['logs'] / 'tensorboard', dirs_exist_ok=True
        )

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
torch.cuda.manual_seed_all(CFG.seed)
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

CONFIG_DICT = asdict(CFG)
CONFIG_HASH = sha256_bytes(stable_json_bytes(CONFIG_DICT))
atomic_write_json(
    RUN_PATHS['config'] / 'run_config.json',
    {'run_id': RUN_ID, 'config': CONFIG_DICT, 'config_sha256': CONFIG_HASH},
)

environment = {
    'run_id': RUN_ID,
    'timestamp_utc': datetime.datetime.now(datetime.UTC).isoformat(),
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'cudnn': torch.backends.cudnn.version(),
    'gpu': GPU_NAME,
    'gpu_vram_gib': GPU_VRAM_GIB,
    'tf32': True,
    'production_precision': 'BF16 autocast with FP32 parameters',
}
atomic_write_json(RUN_PATHS['config'] / 'environment.json', environment)

pip_freeze = subprocess.check_output(
    [sys.executable, '-m', 'pip', 'freeze'], text=True
)
atomic_write_text(RUN_PATHS['config'] / 'pip_freeze.txt', pip_freeze)

# Preserve, but do not automatically use, the legacy D=32 teacher.
legacy_teacher_candidates = [
    DRIVE_ROOT / 'optimized_adjoint_teacher.pt',
    Path('/content/drive/MyDrive/Colab Notebooks/Phase_N3_Logs/optimized_adjoint_teacher.pt'),
]
legacy_teacher_manifest = {'found': False, 'candidates': [str(p) for p in legacy_teacher_candidates]}
for candidate in legacy_teacher_candidates:
    if candidate.exists():
        legacy_teacher_manifest = {
            'found': True,
            'path': str(candidate),
            'bytes': candidate.stat().st_size,
            'sha256': sha256_file(candidate),
            'use_policy': 'read-only audit asset; no semantic compatibility assumed',
        }
        break
atomic_write_json(
    RUN_PATHS['config'] / 'legacy_teacher_manifest.json', legacy_teacher_manifest
)

writer = SummaryWriter(log_dir=str(TB_LOCAL))

print('Run ID:', RUN_ID)
print('Persistent run directory:', RUN_DIR)
print('Legacy teacher:', legacy_teacher_manifest)

In [ ]:
# ================================================================
# 2. Official DROID RLDS access and real visual-feature cache
# ================================================================

import tensorflow as tf
import tensorflow_datasets as tfds

# Prevent TensorFlow from reserving the training GPU.
tf.config.set_visible_devices([], 'GPU')

from torchvision.models import resnet18, ResNet18_Weights
from torchvision.transforms import functional as TVF
import matplotlib.pyplot as plt

SOURCE_MANIFEST = {
    'dataset': CFG.dataset_name,
    'data_dir': CFG.dataset_data_dir,
    'format': 'RLDS / TensorFlow Datasets',
    'purpose': 'Official 100-trajectory DROID prototyping sample',
    'synthetic_fallback_allowed': False,
}
atomic_write_json(RUN_PATHS['config'] / 'source_manifest.json', SOURCE_MANIFEST)
SOURCE_HASH = sha256_bytes(stable_json_bytes(SOURCE_MANIFEST))

try:
    raw_dataset, dataset_info = tfds.load(
        CFG.dataset_name,
        data_dir=CFG.dataset_data_dir,
        split='train',
        shuffle_files=False,
        with_info=True,
    )
except Exception as error:
    raise RuntimeError(
        'Unable to load the official DROID droid_100 RLDS dataset. '
        'No synthetic or unrelated fallback is permitted.'
    ) from error

atomic_write_text(
    RUN_PATHS['config'] / 'tfds_builder_info.txt', str(dataset_info)
)

# Frozen published visual representation target.
visual_weights = ResNet18_Weights.DEFAULT
visual_encoder = resnet18(weights=visual_weights)
visual_encoder.fc = nn.Identity()
visual_encoder.eval().to(DEVICE)
for parameter in visual_encoder.parameters():
    parameter.requires_grad_(False)

IMAGENET_MEAN = torch.tensor(
    [0.485, 0.456, 0.406], device=DEVICE
).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor(
    [0.229, 0.224, 0.225], device=DEVICE
).view(1, 3, 1, 1)

@torch.inference_mode()
def embed_images(images_uint8):
    features = []
    for start in range(0, len(images_uint8), CFG.embedding_batch_size):
        batch_np = np.stack(
            images_uint8[start : start + CFG.embedding_batch_size]
        )
        batch = torch.from_numpy(batch_np).to(DEVICE)
        batch = batch.permute(0, 3, 1, 2).contiguous()
        batch = TVF.resize(
            batch, [CFG.image_size, CFG.image_size], antialias=True
        )
        batch = batch.float().div_(255.0)
        batch = (batch - IMAGENET_MEAN) / IMAGENET_STD
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            encoded = visual_encoder(batch)
        features.append(encoded.float().cpu())
    return torch.cat(features, dim=0).numpy().astype(np.float16)


def numpy_value(value):
    if isinstance(value, np.ndarray):
        return value
    if hasattr(value, 'numpy'):
        return value.numpy()
    return np.asarray(value)


def decode_text(value):
    value = numpy_value(value)
    if isinstance(value, np.ndarray) and value.shape == ():
        value = value.item()
    if isinstance(value, bytes):
        return value.decode('utf-8', errors='replace')
    return str(value)


def nested_value(mapping, path):
    current = mapping
    for key in path:
        if not isinstance(current, dict) or key not in current:
            return None
        current = current[key]
    return current


def choose_episode_id(episode, episode_index):
    candidate_paths = [
        ('episode_metadata', 'recording_folderpath'),
        ('episode_metadata', 'file_path'),
        ('traj_metadata', 'episode_metadata', 'recording_folderpath'),
        ('traj_metadata', 'episode_metadata', 'file_path'),
    ]
    pieces = []
    for path in candidate_paths:
        value = nested_value(episode, path)
        if value is not None:
            pieces.append(decode_text(value))
    if not pieces:
        pieces = [f'droid100_episode_{episode_index:04d}']
    return hashlib.sha256('|'.join(pieces).encode()).hexdigest()[:24]


def extract_state(observation):
    parts, keys = [], []
    for key in ('cartesian_position', 'gripper_position', 'joint_position'):
        if key in observation:
            value = np.asarray(
                numpy_value(observation[key]), dtype=np.float32
            ).reshape(-1)
            parts.append(value)
            keys.append(key)
    if not parts:
        raise KeyError(
            'No supported DROID state fields were present. '
            f'Observation keys: {list(observation.keys())}'
        )
    return np.concatenate(parts), keys

records = []
sample_frames = []
state_keys_contract = None
state_dim_contract = None
action_dim_contract = None

for episode_index, episode in enumerate(raw_dataset.take(CFG.max_episodes)):
    steps = list(tfds.as_numpy(episode['steps']))
    if len(steps) < CFG.context_len + CFG.horizon + 1:
        log_warning(
            'EPISODE_TOO_SHORT',
            'Episode skipped because it is shorter than the temporal contract.',
            episode_index=episode_index,
            length=len(steps),
        )
        continue

    episode_id = choose_episode_id(episode, episode_index)
    states, actions = [], []
    exterior_images, wrist_images = [], []
    instructions = []

    for step_index, step in enumerate(steps):
        if step_index % CFG.frame_stride != 0:
            continue

        observation = step['observation']
        required_images = ('exterior_image_1_left', 'wrist_image_left')
        missing = [key for key in required_images if key not in observation]
        if missing:
            raise KeyError(
                f'Required real image payloads are missing: {missing}. '
                'No hash-only replacement is allowed.'
            )

        exterior = np.asarray(observation['exterior_image_1_left'], dtype=np.uint8)
        wrist = np.asarray(observation['wrist_image_left'], dtype=np.uint8)
        if exterior.ndim != 3 or wrist.ndim != 3:
            raise ValueError('Real image payload has invalid rank.')
        if exterior.var() == 0 or wrist.var() == 0:
            raise ValueError('Real image payload is constant.')

        state, used_state_keys = extract_state(observation)
        action = np.asarray(step['action'], dtype=np.float32).reshape(-1)

        if state_keys_contract is None:
            state_keys_contract = used_state_keys
            state_dim_contract = int(state.shape[0])
            action_dim_contract = int(action.shape[0])
        else:
            assert used_state_keys == state_keys_contract
            assert state.shape[0] == state_dim_contract
            assert action.shape[0] == action_dim_contract

        states.append(state)
        actions.append(action)
        exterior_images.append(exterior)
        wrist_images.append(wrist)
        instructions.append(decode_text(step.get('language_instruction', b'')))

    states = np.stack(states).astype(np.float32)
    actions = np.stack(actions).astype(np.float32)
    exterior_embeddings = embed_images(exterior_images)
    wrist_embeddings = embed_images(wrist_images)

    episode_path = LOCAL_EPISODES / f'{episode_id}.npz'
    np.savez_compressed(
        episode_path,
        states=states,
        actions=actions,
        exterior_embeddings=exterior_embeddings,
        wrist_embeddings=wrist_embeddings,
    )

    instruction = next((item for item in instructions if item.strip()), '')
    record = {
        'episode_index': episode_index,
        'episode_id': episode_id,
        'cached_path': str(episode_path),
        'cached_sha256': sha256_file(episode_path),
        'length': int(len(states)),
        'state_dim': int(states.shape[-1]),
        'action_dim': int(actions.shape[-1]),
        'instruction': instruction,
    }
    records.append(record)

    if len(sample_frames) < 8:
        sample_frames.append(
            {
                'episode_id': episode_id,
                'exterior': exterior_images[0],
                'wrist': wrist_images[0],
                'states': states,
                'actions': actions,
            }
        )

    print(
        f'Cached {len(records):3d}/{CFG.max_episodes}: '
        f'{episode_id}, {len(states)} frames'
    )

if len(records) < 20:
    raise RuntimeError(
        f'Only {len(records)} valid episodes were cached; at least 20 are required.'
    )

# Episode-level split before any window generation.
records_sorted = sorted(
    records,
    key=lambda row: hashlib.sha256(row['episode_id'].encode()).hexdigest(),
)
n_total = len(records_sorted)
n_train = max(1, int(0.80 * n_total))
n_val = max(1, int(0.10 * n_total))

for index, record in enumerate(records_sorted):
    if index < n_train:
        record['split'] = 'train'
    elif index < n_train + n_val:
        record['split'] = 'validation'
    else:
        record['split'] = 'test'

split_sets = {
    split: {
        row['episode_id'] for row in records_sorted if row['split'] == split
    }
    for split in ('train', 'validation', 'test')
}
assert split_sets['train'].isdisjoint(split_sets['validation'])
assert split_sets['train'].isdisjoint(split_sets['test'])
assert split_sets['validation'].isdisjoint(split_sets['test'])

DATA_MANIFEST = {
    'source': SOURCE_MANIFEST,
    'frame_stride': CFG.frame_stride,
    'state_keys': state_keys_contract,
    'state_dim': state_dim_contract,
    'action_dim': action_dim_contract,
    'episodes': records_sorted,
    'split_counts': {split: len(ids) for split, ids in split_sets.items()},
}
DATA_MANIFEST_HASH = sha256_bytes(stable_json_bytes(DATA_MANIFEST))
atomic_write_json(
    RUN_PATHS['config'] / 'data_manifest.json',
    {**DATA_MANIFEST, 'data_manifest_sha256': DATA_MANIFEST_HASH},
)

fig, axes = plt.subplots(
    len(sample_frames), 3, figsize=(14, 3 * len(sample_frames))
)
if len(sample_frames) == 1:
    axes = np.expand_dims(axes, axis=0)

for row_index, sample in enumerate(sample_frames):
    axes[row_index, 0].imshow(sample['exterior'])
    axes[row_index, 0].set_title(f"{sample['episode_id']} exterior")
    axes[row_index, 1].imshow(sample['wrist'])
    axes[row_index, 1].set_title('wrist')
    axes[row_index, 2].plot(sample['states'][:, 0], label='state[0]')
    axes[row_index, 2].plot(sample['actions'][:, 0], label='action[0]')
    axes[row_index, 2].legend()
    axes[row_index, 2].set_title('trajectory channels')
    axes[row_index, 0].axis('off')
    axes[row_index, 1].axis('off')

plt.tight_layout()
fig.savefig(RUN_PATHS['samples'] / 'episode_audit.png', dpi=150)
plt.show()
plt.close(fig)

DATA_AUDIT = {
    'valid_episode_count': len(records_sorted),
    'split_counts': DATA_MANIFEST['split_counts'],
    'state_keys': state_keys_contract,
    'state_dim': state_dim_contract,
    'action_dim': action_dim_contract,
    'real_image_payloads_verified': True,
    'split_disjointness_verified': True,
    'synthetic_fallback_used': False,
    'status': 'PASS',
}
atomic_write_json(RUN_PATHS['tests'] / 'data_audit.json', DATA_AUDIT)
print(json.dumps(DATA_AUDIT, indent=2))

In [ ]:
# ================================================================
# 3. Temporal windows, normalisation, and recursive world model
# ================================================================

from functools import lru_cache

train_records = [row for row in records_sorted if row['split'] == 'train']
val_records = [row for row in records_sorted if row['split'] == 'validation']
test_records = [row for row in records_sorted if row['split'] == 'test']


def compute_train_statistics(records):
    states, actions = [], []
    for record in records:
        with np.load(record['cached_path']) as episode:
            states.append(episode['states'].astype(np.float64))
            actions.append(episode['actions'].astype(np.float64))
    states = np.concatenate(states, axis=0)
    actions = np.concatenate(actions, axis=0)
    return {
        'state_mean': states.mean(axis=0).astype(np.float32),
        'state_std': np.maximum(states.std(axis=0), 1e-6).astype(np.float32),
        'action_mean': actions.mean(axis=0).astype(np.float32),
        'action_std': np.maximum(actions.std(axis=0), 1e-6).astype(np.float32),
    }

NORMALISATION = compute_train_statistics(train_records)
np.savez(RUN_PATHS['config'] / 'normalisation.npz', **NORMALISATION)


class WindowDataset(Dataset):
    def __init__(self, records, context_len, horizon, stride, normalisation):
        self.records = records
        self.context_len = context_len
        self.horizon = horizon
        self.stride = stride
        self.normalisation = normalisation
        self.index = []
        for record_index, record in enumerate(records):
            maximum_start = record['length'] - context_len - horizon
            for start in range(0, maximum_start + 1, stride):
                self.index.append((record_index, start))
        if not self.index:
            raise RuntimeError('No temporal windows satisfy the contract.')

    def __len__(self):
        return len(self.index)

    @lru_cache(maxsize=16)
    def _load_episode(self, path):
        with np.load(path) as episode:
            return {key: episode[key].copy() for key in episode.files}

    def __getitem__(self, item):
        record_index, start = self.index[item]
        record = self.records[record_index]
        episode = self._load_episode(record['cached_path'])

        states = episode['states'].astype(np.float32)
        actions = episode['actions'].astype(np.float32)
        visual = np.concatenate(
            [
                episode['exterior_embeddings'].astype(np.float32),
                episode['wrist_embeddings'].astype(np.float32),
            ],
            axis=-1,
        )

        states_norm = (
            states - self.normalisation['state_mean']
        ) / self.normalisation['state_std']
        actions_norm = (
            actions - self.normalisation['action_mean']
        ) / self.normalisation['action_std']

        context_end = start + self.context_len
        context_state = states_norm[start:context_end]
        context_visual = visual[start:context_end]

        # action[t] is aligned with the transition from state[t] toward state[t+1].
        context_action = np.zeros(
            (self.context_len, actions.shape[-1]), dtype=np.float32
        )
        if self.context_len > 1:
            context_action[1:] = actions_norm[start : context_end - 1]

        future_actions = actions_norm[
            context_end - 1 : context_end - 1 + self.horizon
        ]
        target_state = states_norm[context_end : context_end + self.horizon]
        target_visual = visual[context_end : context_end + self.horizon]

        assert len(future_actions) == self.horizon
        assert len(target_state) == self.horizon
        assert len(target_visual) == self.horizon

        return {
            'context_state': torch.from_numpy(context_state),
            'context_action': torch.from_numpy(context_action),
            'context_visual': torch.from_numpy(context_visual),
            'future_actions': torch.from_numpy(future_actions),
            'target_state': torch.from_numpy(target_state),
            'target_visual': torch.from_numpy(target_visual),
            'episode_id': record['episode_id'],
            'window_start': start,
        }

train_dataset = WindowDataset(
    train_records, CFG.context_len, CFG.horizon, CFG.window_stride, NORMALISATION
)
val_dataset = WindowDataset(
    val_records, CFG.context_len, CFG.horizon, CFG.window_stride, NORMALISATION
)
test_dataset = WindowDataset(
    test_records, CFG.context_len, CFG.horizon, CFG.window_stride, NORMALISATION
)
print('Windows:', len(train_dataset), len(val_dataset), len(test_dataset))


class StatefulBatchSampler(Sampler):
    def __init__(self, length, batch_size, seed):
        self.length = int(length)
        self.batch_size = int(batch_size)
        self.seed = int(seed)
        self.epoch = 0
        self.cursor = 0
        self.permutation = None
        self._regenerate()

    def _regenerate(self):
        generator = torch.Generator()
        generator.manual_seed(self.seed + self.epoch)
        self.permutation = torch.randperm(
            self.length, generator=generator
        ).tolist()
        self.cursor = 0

    def __iter__(self):
        while self.cursor < self.length:
            end = min(self.cursor + self.batch_size, self.length)
            batch = self.permutation[self.cursor:end]
            self.cursor = end
            yield batch
        self.epoch += 1
        self._regenerate()

    def __len__(self):
        return math.ceil((self.length - self.cursor) / self.batch_size)

    def state_dict(self):
        return {
            'length': self.length,
            'batch_size': self.batch_size,
            'seed': self.seed,
            'epoch': self.epoch,
            'cursor': self.cursor,
            'permutation': self.permutation,
        }

    def load_state_dict(self, state):
        assert state['length'] == self.length
        assert state['batch_size'] == self.batch_size
        self.seed = int(state['seed'])
        self.epoch = int(state['epoch'])
        self.cursor = int(state['cursor'])
        self.permutation = list(state['permutation'])


class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, value):
        return self.net(value)


class AdjointRecursiveWorldModel(nn.Module):
    def __init__(self, state_dim, action_dim, visual_dim, config):
        super().__init__()
        d = config.d_model
        self.d_model = d
        self.horizon = config.horizon
        self.num_candidates = config.num_refinement_candidates

        self.visual_adapter = nn.Sequential(
            nn.LayerNorm(visual_dim), nn.Linear(visual_dim, d), nn.GELU()
        )
        self.state_adapter = nn.Sequential(
            nn.Linear(state_dim, d), nn.LayerNorm(d), nn.GELU()
        )
        self.context_action_adapter = nn.Sequential(
            nn.Linear(action_dim, d), nn.LayerNorm(d), nn.GELU()
        )
        self.future_action_adapter = nn.Sequential(
            nn.Linear(action_dim, d), nn.LayerNorm(d), nn.GELU()
        )

        self.position = nn.Parameter(torch.zeros(1, config.context_len, d))
        nn.init.normal_(self.position, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d,
            nhead=config.transformer_heads,
            dim_feedforward=config.transformer_ff,
            dropout=config.dropout,
            batch_first=True,
            norm_first=True,
            activation='gelu',
        )
        self.temporal_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=config.transformer_layers
        )
        self.temporal_norm = nn.LayerNorm(d)

        self.candidate_embedding = nn.Embedding(self.num_candidates, d)
        self.shared_refiner = MLP(2 * d, 4 * d, d, dropout=config.dropout)

        self.transition = nn.GRUCell(d, d)
        self.state_mean_head = nn.Linear(d, state_dim)
        self.state_logvar_head = nn.Linear(d, state_dim)
        self.visual_head = nn.Linear(d, visual_dim)

        self.objective_condition = MLP(d + 2, 2 * d, d)
        self.costate_head = nn.Linear(d, d)
        self.critic_head = MLP(2 * d + 1, 2 * d, 1)
        self.gate_head = MLP(d, d, 1)

        self.register_buffer(
            'candidate_costs',
            torch.tensor(config.candidate_costs, dtype=torch.float32),
        )

    def encode_context(self, context_visual, context_state, context_action):
        tokens = (
            self.visual_adapter(context_visual)
            + self.state_adapter(context_state)
            + self.context_action_adapter(context_action)
        )
        tokens = tokens + self.position[:, : tokens.shape[1]]
        encoded = self.temporal_encoder(tokens)
        return self.temporal_norm(encoded[:, -1])

    def candidate_effects(self, latent):
        batch = latent.shape[0]
        candidates = self.candidate_embedding.weight.unsqueeze(0).expand(
            batch, -1, -1
        )
        latent_expanded = latent.unsqueeze(1).expand(
            -1, self.num_candidates, -1
        )
        inputs = torch.cat([latent_expanded, candidates], dim=-1)
        return 0.10 * torch.tanh(self.shared_refiner(inputs))

    def apply_mask(self, latent, effects, mask):
        return latent + (effects * mask.unsqueeze(-1)).sum(dim=1)

    def rollout(self, latent, future_actions):
        states, logvars, visuals = [], [], []
        current = latent
        for horizon_index in range(future_actions.shape[1]):
            action = self.future_action_adapter(
                future_actions[:, horizon_index]
            )
            current = self.transition(action, current)
            states.append(self.state_mean_head(current))
            logvars.append(
                torch.clamp(self.state_logvar_head(current), -6.0, 3.0)
            )
            visuals.append(self.visual_head(current))
        return {
            'state_mean': torch.stack(states, dim=1),
            'state_logvar': torch.stack(logvars, dim=1),
            'visual': torch.stack(visuals, dim=1),
        }

    def predict_costate(self, latent, budget_fraction, horizon_fraction):
        condition = torch.cat(
            [
                latent,
                budget_fraction.view(-1, 1),
                horizon_fraction.view(-1, 1),
            ],
            dim=-1,
        )
        return self.costate_head(self.objective_condition(condition))

    def predict_critic(self, latent, effects):
        batch, candidates, _ = effects.shape
        latent_expanded = latent.unsqueeze(1).expand(-1, candidates, -1)
        costs = self.candidate_costs.view(1, candidates, 1).expand(
            batch, -1, -1
        )
        critic_input = torch.cat([latent_expanded, effects, costs], dim=-1)
        return self.critic_head(critic_input).squeeze(-1)

    def predict_gate(self, latent):
        return self.gate_head(latent).squeeze(-1)

state_dim = DATA_MANIFEST['state_dim']
action_dim = DATA_MANIFEST['action_dim']
visual_dim = 2 * CFG.visual_embedding_dim_per_camera

model = AdjointRecursiveWorldModel(
    state_dim=state_dim,
    action_dim=action_dim,
    visual_dim=visual_dim,
    config=CFG,
).to(DEVICE)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)
print(f'Trainable parameters: {trainable_parameters:,}')

MODEL_MANIFEST = {
    'class': 'AdjointRecursiveWorldModel',
    'trainable_parameters': trainable_parameters,
    'state_dim': state_dim,
    'action_dim': action_dim,
    'visual_dim': visual_dim,
    'd_model': CFG.d_model,
    'recursive_candidates': CFG.num_refinement_candidates,
    'shared_refiner': True,
    'visual_target': 'frozen ResNet-18 ImageNet weights',
}
atomic_write_json(RUN_PATHS['config'] / 'model_manifest.json', MODEL_MANIFEST)

with torch.no_grad():
    example = default_collate([train_dataset[0], train_dataset[1]])
    z = model.encode_context(
        example['context_visual'].to(DEVICE),
        example['context_state'].to(DEVICE),
        example['context_action'].to(DEVICE),
    )
    effects = model.candidate_effects(z)
    output = model.rollout(z, example['future_actions'].to(DEVICE))

assert z.shape == (2, CFG.d_model)
assert effects.shape == (2, CFG.num_refinement_candidates, CFG.d_model)
assert output['state_mean'].shape == (2, CFG.horizon, state_dim)
print('Model shape test: PASS')

In [ ]:
# ================================================================
# 4. Dynamics objective, baselines, benchmark, and Stage 1 training
# ================================================================


def variance_covariance_regularizer(latent):
    latent = latent.float()
    latent = latent - latent.mean(dim=0, keepdim=True)
    std = torch.sqrt(latent.var(dim=0, unbiased=False) + 1e-4)
    variance_loss = F.relu(1.0 - std).mean()
    if latent.shape[0] < 2:
        return variance_loss
    covariance = latent.T @ latent / max(1, latent.shape[0] - 1)
    diagonal = torch.diag(torch.diag(covariance))
    covariance_loss = (covariance - diagonal).pow(2).sum() / latent.shape[1]
    return variance_loss + 0.01 * covariance_loss


def objective_per_sample(predictions, target_state, target_visual):
    mean = predictions['state_mean'].float()
    logvar = predictions['state_logvar'].float()
    target_state = target_state.float()

    state_nll = 0.5 * (
        torch.exp(-logvar) * (mean - target_state).pow(2) + logvar
    )
    state_nll = state_nll.mean(dim=(1, 2))

    predicted_visual = F.normalize(predictions['visual'].float(), dim=-1)
    target_visual = F.normalize(target_visual.float(), dim=-1)
    visual_loss = (
        1.0 - (predicted_visual * target_visual).sum(dim=-1)
    ).mean(dim=1)
    return state_nll + 0.25 * visual_loss


def sample_candidate_mask(batch_size, num_candidates, device):
    budgets = torch.randint(
        low=0,
        high=num_candidates + 1,
        size=(batch_size,),
        device=device,
    )
    random_scores = torch.rand(batch_size, num_candidates, device=device)
    ordering = torch.argsort(random_scores, dim=1, descending=True)
    mask = torch.zeros_like(random_scores)
    for row in range(batch_size):
        count = int(budgets[row].item())
        if count:
            mask[row, ordering[row, :count]] = 1.0
    return mask, budgets


def stage1_loss(model, batch):
    context_visual = batch['context_visual'].to(DEVICE, non_blocking=True)
    context_state = batch['context_state'].to(DEVICE, non_blocking=True)
    context_action = batch['context_action'].to(DEVICE, non_blocking=True)
    future_actions = batch['future_actions'].to(DEVICE, non_blocking=True)
    target_state = batch['target_state'].to(DEVICE, non_blocking=True)
    target_visual = batch['target_visual'].to(DEVICE, non_blocking=True)

    latent = model.encode_context(context_visual, context_state, context_action)
    effects = model.candidate_effects(latent)

    mask, budgets = sample_candidate_mask(
        latent.shape[0], model.num_candidates, DEVICE
    )
    full_mask = torch.ones_like(mask)
    zero_mask = torch.zeros_like(mask)

    allocated_prediction = model.rollout(
        model.apply_mask(latent, effects, mask), future_actions
    )
    full_prediction = model.rollout(
        model.apply_mask(latent, effects, full_mask), future_actions
    )
    base_prediction = model.rollout(
        model.apply_mask(latent, effects, zero_mask), future_actions
    )

    objective_allocated = objective_per_sample(
        allocated_prediction, target_state, target_visual
    )
    objective_full = objective_per_sample(
        full_prediction, target_state, target_visual
    )
    objective_base = objective_per_sample(
        base_prediction, target_state, target_visual
    )

    rate_cost = (mask * model.candidate_costs.view(1, -1)).sum(dim=1)

    normalized_effects = F.normalize(effects.float(), dim=-1)
    gram = normalized_effects @ normalized_effects.transpose(1, 2)
    eye = torch.eye(model.num_candidates, device=DEVICE).unsqueeze(0)
    diversity_loss = (gram - eye).pow(2).mean()
    collapse_loss = variance_covariance_regularizer(latent)

    loss = (
        objective_allocated.mean()
        + 0.25 * objective_base.mean()
        + 0.50 * objective_full.mean()
        + CFG.rate_beta * rate_cost.mean()
        + 0.01 * diversity_loss
        + 0.01 * collapse_loss
    )

    return loss, {
        'objective_allocated': objective_allocated.mean().detach(),
        'objective_base': objective_base.mean().detach(),
        'objective_full': objective_full.mean().detach(),
        'rate_cost': rate_cost.mean().detach(),
        'diversity': diversity_loss.detach(),
        'collapse': collapse_loss.detach(),
        'latent_std': latent.float().std().detach(),
        'mean_budget': budgets.float().mean().detach(),
    }


@torch.no_grad()
def evaluate_dynamics(model, dataset, batch_size):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )
    squared_error_full = torch.zeros(CFG.horizon, device=DEVICE)
    squared_error_base = torch.zeros(CFG.horizon, device=DEVICE)
    squared_error_persistence = torch.zeros(CFG.horizon, device=DEVICE)
    count = 0
    visual_cosine = 0.0

    for batch in loader:
        context_visual = batch['context_visual'].to(DEVICE)
        context_state = batch['context_state'].to(DEVICE)
        context_action = batch['context_action'].to(DEVICE)
        future_actions = batch['future_actions'].to(DEVICE)
        target_state = batch['target_state'].to(DEVICE)
        target_visual = batch['target_visual'].to(DEVICE)

        latent = model.encode_context(context_visual, context_state, context_action)
        effects = model.candidate_effects(latent)
        full = model.rollout(latent + effects.sum(dim=1), future_actions)
        base = model.rollout(latent, future_actions)
        persistence = context_state[:, -1:].expand(-1, CFG.horizon, -1)

        squared_error_full += (
            (full['state_mean'] - target_state).pow(2).mean(dim=(0, 2))
        )
        squared_error_base += (
            (base['state_mean'] - target_state).pow(2).mean(dim=(0, 2))
        )
        squared_error_persistence += (
            (persistence - target_state).pow(2).mean(dim=(0, 2))
        )
        visual_cosine += F.cosine_similarity(
            full['visual'], target_visual, dim=-1
        ).mean().item()
        count += 1

    return {
        'full_rmse_by_horizon': torch.sqrt(
            squared_error_full / count
        ).cpu().tolist(),
        'base_rmse_by_horizon': torch.sqrt(
            squared_error_base / count
        ).cpu().tolist(),
        'persistence_rmse_by_horizon': torch.sqrt(
            squared_error_persistence / count
        ).cpu().tolist(),
        'visual_cosine': visual_cosine / count,
    }


def capture_rng_state():
    return {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
        'torch_cuda': torch.cuda.get_rng_state_all(),
    }


def restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch_cpu'])
    torch.cuda.set_rng_state_all(state['torch_cuda'])


def save_checkpoint(
    name, stage, model, optimizer, scheduler, sampler, step, best_metrics
):
    local_path = LOCAL_WORK / name
    drive_path = RUN_PATHS['checkpoints'] / name
    payload = {
        'schema_version': 2,
        'run_id': RUN_ID,
        'stage': stage,
        'global_step': int(step),
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': (
            optimizer.state_dict() if optimizer is not None else None
        ),
        'scheduler_state_dict': (
            scheduler.state_dict() if scheduler is not None else None
        ),
        'sampler_state': sampler.state_dict() if sampler is not None else None,
        'rng_state': capture_rng_state(),
        'config': CONFIG_DICT,
        'config_hash': CONFIG_HASH,
        'data_manifest_hash': DATA_MANIFEST_HASH,
        'source_hash': SOURCE_HASH,
        'normalisation': NORMALISATION,
        'best_metrics': best_metrics,
        'environment': environment,
    }
    torch.save(payload, local_path)
    digest = atomic_copy(local_path, drive_path)
    return {'path': str(drive_path), 'sha256': digest}


def warmup_cosine_multiplier(step):
    if step < CFG.warmup_steps:
        return (step + 1) / max(1, CFG.warmup_steps)
    progress = (step - CFG.warmup_steps) / max(
        1, CFG.dynamics_steps - CFG.warmup_steps
    )
    return 0.5 * (1.0 + math.cos(math.pi * progress))

# Forward+backward benchmark on the actual training objective.
benchmark_rows = []
for batch_size in CFG.microbatch_candidates:
    try:
        loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
        )
        batch = next(iter(loader))
        benchmark_model = copy.deepcopy(model).to(DEVICE)
        benchmark_model.train()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        for _ in range(3):
            benchmark_model.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                loss, _ = stage1_loss(benchmark_model, batch)
            loss.backward()

        torch.cuda.synchronize()
        start = time.perf_counter()
        iterations = 10
        for _ in range(iterations):
            benchmark_model.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                loss, _ = stage1_loss(benchmark_model, batch)
            loss.backward()
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        benchmark_rows.append(
            {
                'batch_size': batch_size,
                'train_steps_per_second': iterations / elapsed,
                'sequences_per_second': iterations * batch_size / elapsed,
                'peak_vram_gib': torch.cuda.max_memory_allocated() / 1024**3,
                'status': 'PASS',
            }
        )
        del benchmark_model
        torch.cuda.empty_cache()
    except torch.cuda.OutOfMemoryError:
        benchmark_rows.append({'batch_size': batch_size, 'status': 'OOM'})
        torch.cuda.empty_cache()

benchmark_frame = pd.DataFrame(benchmark_rows)
benchmark_frame.to_csv(
    RUN_PATHS['benchmarks'] / 'microbatch_training_benchmark.csv', index=False
)
display(benchmark_frame)

valid_benchmarks = benchmark_frame[benchmark_frame['status'] == 'PASS'].copy()
if valid_benchmarks.empty:
    raise RuntimeError('No microbatch fits on the active GPU.')

peak_throughput = valid_benchmarks['sequences_per_second'].max()
near_peak = valid_benchmarks[
    valid_benchmarks['sequences_per_second'] >= 0.95 * peak_throughput
]
SELECTED_BATCH_SIZE = int(
    near_peak.sort_values('batch_size').iloc[0]['batch_size']
)
print('Selected microbatch:', SELECTED_BATCH_SIZE)

train_sampler = StatefulBatchSampler(
    len(train_dataset), SELECTED_BATCH_SIZE, CFG.seed
)
train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_sampler,
    num_workers=0,
    pin_memory=True,
)
train_iterator = iter(train_loader)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.learning_rate,
    weight_decay=CFG.weight_decay,
)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=warmup_cosine_multiplier
)

best_validation = float('inf')
best_metrics = {}
best_checkpoint = None
model.train()

for step in range(1, CFG.dynamics_steps + 1):
    try:
        batch = next(train_iterator)
    except StopIteration:
        train_iterator = iter(train_loader)
        batch = next(train_iterator)

    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        loss, components = stage1_loss(model, batch)

    if not torch.isfinite(loss):
        raise FloatingPointError(f'Non-finite dynamics loss at step {step}')

    loss.backward()
    gradient_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(), CFG.gradient_clip
    )
    if not torch.isfinite(gradient_norm):
        raise FloatingPointError(f'Non-finite dynamics gradient at step {step}')

    optimizer.step()
    scheduler.step()

    writer.add_scalar('dynamics/train_loss', loss.item(), step)
    writer.add_scalar('dynamics/gradient_norm', float(gradient_norm), step)

    if step == 1 or step % 25 == 0:
        record = log_metric(
            'dynamics_train',
            step,
            loss=float(loss.item()),
            gradient_norm=float(gradient_norm),
            learning_rate=optimizer.param_groups[0]['lr'],
            **{key: float(value) for key, value in components.items()},
        )
        print(record)

    if step % CFG.eval_interval == 0 or step == CFG.dynamics_steps:
        model.eval()
        validation = evaluate_dynamics(
            model, val_dataset, SELECTED_BATCH_SIZE
        )
        model.train()
        validation_score = float(
            np.mean(validation['full_rmse_by_horizon'])
        )
        log_metric(
            'dynamics_validation',
            step,
            validation_score=validation_score,
            **validation,
        )
        writer.add_scalar('dynamics/validation_rmse', validation_score, step)

        if validation_score < best_validation:
            best_validation = validation_score
            best_metrics = validation
            best_checkpoint = save_checkpoint(
                'best_dynamics.pt',
                'dynamics',
                model,
                optimizer,
                scheduler,
                train_sampler,
                step,
                best_metrics,
            )
        sync_runtime_logs()

    if step % CFG.checkpoint_interval == 0:
        save_checkpoint(
            'latest_dynamics.pt',
            'dynamics',
            model,
            optimizer,
            scheduler,
            train_sampler,
            step,
            best_metrics,
        )

if best_checkpoint is None:
    raise RuntimeError('No dynamics checkpoint was promoted.')

dynamics_payload = torch.load(
    best_checkpoint['path'], map_location=DEVICE, weights_only=False
)
model.load_state_dict(dynamics_payload['model_state_dict'], strict=True)

test_dynamics = evaluate_dynamics(model, test_dataset, SELECTED_BATCH_SIZE)
full_rmse = np.mean(test_dynamics['full_rmse_by_horizon'])
persistence_rmse = np.mean(test_dynamics['persistence_rmse_by_horizon'])
relative_improvement = (
    persistence_rmse - full_rmse
) / max(persistence_rmse, 1e-8)

DYNAMICS_GATE = {
    'full_rmse': float(full_rmse),
    'persistence_rmse': float(persistence_rmse),
    'relative_improvement': float(relative_improvement),
    'required_margin': CFG.baseline_improvement_margin,
    'passed': bool(relative_improvement >= CFG.baseline_improvement_margin),
    'visual_cosine': test_dynamics['visual_cosine'],
}
atomic_write_json(
    RUN_PATHS['artifacts'] / 'dynamics_evaluation.json',
    {'test': test_dynamics, 'gate': DYNAMICS_GATE},
)
print(json.dumps(DYNAMICS_GATE, indent=2))

if not DYNAMICS_GATE['passed']:
    raise RuntimeError(
        'Dynamics model did not beat persistence by the frozen margin. '
        'Adjoint training is blocked.'
    )

In [ ]:
# ================================================================
# 5. Exact training-only co-states, matched critic, and allocator
# ================================================================

teacher = copy.deepcopy(model).eval().to(DEVICE)
for parameter in teacher.parameters():
    parameter.requires_grad_(False)


def prepare_batch(batch):
    return {
        key: (
            value.to(DEVICE, non_blocking=True)
            if torch.is_tensor(value)
            else value
        )
        for key, value in batch.items()
    }


def exact_adjoint_targets(batch):
    batch = prepare_batch(batch)
    with torch.no_grad():
        latent_initial = teacher.encode_context(
            batch['context_visual'],
            batch['context_state'],
            batch['context_action'],
        )
        effects = teacher.candidate_effects(latent_initial).detach()

    latent = latent_initial.detach().requires_grad_(True)
    stop_prediction = teacher.rollout(latent, batch['future_actions'])
    stop_objective = objective_per_sample(
        stop_prediction, batch['target_state'], batch['target_visual']
    )
    exact_costate = torch.autograd.grad(
        stop_objective.sum(),
        latent,
        create_graph=False,
        retain_graph=False,
    )[0].detach()

    candidate_objectives = []
    with torch.no_grad():
        for candidate_index in range(teacher.num_candidates):
            candidate_latent = latent_initial + effects[:, candidate_index]
            candidate_prediction = teacher.rollout(
                candidate_latent, batch['future_actions']
            )
            candidate_objectives.append(
                objective_per_sample(
                    candidate_prediction,
                    batch['target_state'],
                    batch['target_visual'],
                )
            )

    candidate_objectives = torch.stack(candidate_objectives, dim=1)
    costs = (
        teacher.candidate_costs.view(1, -1) * CFG.candidate_cost_weight
    )
    exact_gain = (
        stop_objective.detach().unsqueeze(1)
        - candidate_objectives
        - costs
    )
    return {
        'batch': batch,
        'latent': latent_initial.detach(),
        'effects': effects,
        'exact_costate': exact_costate,
        'exact_gain': exact_gain,
        'stop_objective': stop_objective.detach(),
    }

# Freeze dynamics and refinement semantics while fitting allocator heads.
for parameter in model.parameters():
    parameter.requires_grad_(False)

head_modules = [
    model.objective_condition,
    model.costate_head,
    model.critic_head,
    model.gate_head,
]
for module in head_modules:
    for parameter in module.parameters():
        parameter.requires_grad_(True)

head_parameters = [
    parameter
    for module in head_modules
    for parameter in module.parameters()
]

allocator_optimizer = torch.optim.AdamW(
    head_parameters,
    lr=CFG.allocator_learning_rate,
    weight_decay=CFG.weight_decay,
)
allocator_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    allocator_optimizer, T_max=CFG.allocator_steps
)

allocator_sampler = StatefulBatchSampler(
    len(train_dataset), SELECTED_BATCH_SIZE, CFG.seed + 100000
)
allocator_loader = DataLoader(
    train_dataset,
    batch_sampler=allocator_sampler,
    num_workers=0,
    pin_memory=True,
)
allocator_iterator = iter(allocator_loader)


def rank_values(values):
    ordering = torch.argsort(values, dim=1)
    return torch.argsort(ordering, dim=1).float()


def mean_batch_spearman(left, right):
    left_rank = rank_values(left)
    right_rank = rank_values(right)
    left_rank = left_rank - left_rank.mean(dim=1, keepdim=True)
    right_rank = right_rank - right_rank.mean(dim=1, keepdim=True)
    numerator = (left_rank * right_rank).sum(dim=1)
    denominator = torch.sqrt(
        left_rank.pow(2).sum(dim=1) * right_rank.pow(2).sum(dim=1)
    ).clamp_min(1e-8)
    return (numerator / denominator).mean()


def allocator_training_loss(targets):
    latent = targets['latent']
    effects = targets['effects']
    exact_costate = targets['exact_costate']
    exact_gain = targets['exact_gain']

    batch_size = latent.shape[0]
    budget_fraction = torch.full(
        (batch_size,), 1.0 / model.num_candidates, device=DEVICE
    )
    horizon_fraction = torch.ones(batch_size, device=DEVICE)

    predicted_costate = model.predict_costate(
        latent, budget_fraction, horizon_fraction
    )
    critic_scores = model.predict_critic(latent, effects)
    adjoint_scores = -(
        predicted_costate.unsqueeze(1) * effects
    ).sum(dim=-1)
    adjoint_scores = (
        adjoint_scores
        - model.candidate_costs.view(1, -1) * CFG.candidate_cost_weight
    )

    normalized_exact = F.normalize(exact_costate, dim=-1)
    normalized_predicted = F.normalize(predicted_costate, dim=-1)
    costate_direction_loss = (
        1.0 - (normalized_exact * normalized_predicted).sum(dim=-1)
    ).mean()

    exact_norm = exact_costate.norm(dim=-1).clamp_min(1e-8)
    predicted_norm = predicted_costate.norm(dim=-1).clamp_min(1e-8)
    costate_magnitude_loss = F.smooth_l1_loss(
        torch.log(predicted_norm), torch.log(exact_norm)
    )

    critic_regression_loss = F.smooth_l1_loss(critic_scores, exact_gain)
    best_candidate = exact_gain.argmax(dim=1)
    critic_ranking_loss = F.cross_entropy(critic_scores, best_candidate)
    adjoint_ranking_loss = F.cross_entropy(adjoint_scores, best_candidate)

    oracle_gain = exact_gain.max(dim=1).values
    adjoint_choice = adjoint_scores.argmax(dim=1)
    critic_choice = critic_scores.argmax(dim=1)
    adjoint_selected_gain = exact_gain.gather(
        1, adjoint_choice.unsqueeze(1)
    ).squeeze(1)
    critic_selected_gain = exact_gain.gather(
        1, critic_choice.unsqueeze(1)
    ).squeeze(1)
    adjoint_regret = oracle_gain - adjoint_selected_gain
    critic_regret = oracle_gain - critic_selected_gain

    gate_target = (
        critic_regret - adjoint_regret > CFG.candidate_cost_weight
    ).float()
    gate_logits = model.predict_gate(latent)
    gate_loss = F.binary_cross_entropy_with_logits(
        gate_logits, gate_target
    )

    loss = (
        costate_direction_loss
        + 0.10 * costate_magnitude_loss
        + critic_regression_loss
        + 0.50 * critic_ranking_loss
        + 0.50 * adjoint_ranking_loss
        + 0.25 * gate_loss
    )

    return loss, {
        'costate_direction_loss': costate_direction_loss.detach(),
        'costate_magnitude_loss': costate_magnitude_loss.detach(),
        'critic_regression_loss': critic_regression_loss.detach(),
        'critic_ranking_loss': critic_ranking_loss.detach(),
        'adjoint_ranking_loss': adjoint_ranking_loss.detach(),
        'gate_loss': gate_loss.detach(),
        'adjoint_regret': adjoint_regret.mean().detach(),
        'critic_regret': critic_regret.mean().detach(),
        'adjoint_spearman': mean_batch_spearman(
            adjoint_scores.detach(), exact_gain.detach()
        ).detach(),
        'critic_spearman': mean_batch_spearman(
            critic_scores.detach(), exact_gain.detach()
        ).detach(),
    }


@torch.no_grad()
def evaluate_allocator(model, dataset, batch_size):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )
    totals = {
        'samples': 0,
        'adjoint_regret': 0.0,
        'critic_regret': 0.0,
        'hybrid_regret': 0.0,
        'random_regret': 0.0,
        'adjoint_top1': 0.0,
        'critic_top1': 0.0,
        'hybrid_top1': 0.0,
        'invocation_count': 0.0,
    }
    traces = []

    for batch in loader:
        with torch.enable_grad():
            targets = exact_adjoint_targets(batch)

        latent = targets['latent']
        effects = targets['effects']
        exact_gain = targets['exact_gain']
        batch_size_actual = latent.shape[0]

        budget_fraction = torch.full(
            (batch_size_actual,),
            1.0 / model.num_candidates,
            device=DEVICE,
        )
        horizon_fraction = torch.ones(batch_size_actual, device=DEVICE)
        predicted_costate = model.predict_costate(
            latent, budget_fraction, horizon_fraction
        )
        critic_scores = model.predict_critic(latent, effects)
        adjoint_scores = -(
            predicted_costate.unsqueeze(1) * effects
        ).sum(dim=-1)
        adjoint_scores = (
            adjoint_scores
            - model.candidate_costs.view(1, -1) * CFG.candidate_cost_weight
        )
        gate_probability = torch.sigmoid(model.predict_gate(latent))

        oracle_choice = exact_gain.argmax(dim=1)
        oracle_gain = exact_gain.max(dim=1).values
        adjoint_choice = adjoint_scores.argmax(dim=1)
        critic_choice = critic_scores.argmax(dim=1)
        hybrid_choice = torch.where(
            gate_probability >= 0.5, adjoint_choice, critic_choice
        )
        random_choice = torch.rand_like(exact_gain).argmax(dim=1)

        def selected_gain(choice):
            return exact_gain.gather(1, choice.unsqueeze(1)).squeeze(1)

        adjoint_gain = selected_gain(adjoint_choice)
        critic_gain = selected_gain(critic_choice)
        hybrid_gain = selected_gain(hybrid_choice)
        random_gain = selected_gain(random_choice)

        totals['samples'] += batch_size_actual
        totals['adjoint_regret'] += (oracle_gain - adjoint_gain).sum().item()
        totals['critic_regret'] += (oracle_gain - critic_gain).sum().item()
        totals['hybrid_regret'] += (oracle_gain - hybrid_gain).sum().item()
        totals['random_regret'] += (oracle_gain - random_gain).sum().item()
        totals['adjoint_top1'] += (adjoint_choice == oracle_choice).sum().item()
        totals['critic_top1'] += (critic_choice == oracle_choice).sum().item()
        totals['hybrid_top1'] += (hybrid_choice == oracle_choice).sum().item()
        totals['invocation_count'] += (
            gate_probability >= 0.5
        ).sum().item()

        for row in range(batch_size_actual):
            traces.append(
                {
                    'episode_id': batch['episode_id'][row],
                    'window_start': int(batch['window_start'][row]),
                    'oracle_choice': int(oracle_choice[row]),
                    'adjoint_choice': int(adjoint_choice[row]),
                    'critic_choice': int(critic_choice[row]),
                    'hybrid_choice': int(hybrid_choice[row]),
                    'gate_probability': float(gate_probability[row]),
                    'adjoint_regret': float(
                        oracle_gain[row] - adjoint_gain[row]
                    ),
                    'critic_regret': float(
                        oracle_gain[row] - critic_gain[row]
                    ),
                    'hybrid_regret': float(
                        oracle_gain[row] - hybrid_gain[row]
                    ),
                }
            )

    samples = totals.pop('samples')
    metrics = {key: value / samples for key, value in totals.items()}
    return metrics, traces

best_hybrid_regret = float('inf')
best_allocator_metrics = {}
best_allocator_checkpoint = None
model.train()

for step in range(1, CFG.allocator_steps + 1):
    try:
        batch = next(allocator_iterator)
    except StopIteration:
        allocator_iterator = iter(allocator_loader)
        batch = next(allocator_iterator)

    allocator_optimizer.zero_grad(set_to_none=True)
    with torch.enable_grad():
        targets = exact_adjoint_targets(batch)
        loss, metrics = allocator_training_loss(targets)

    if not torch.isfinite(loss):
        raise FloatingPointError(f'Non-finite allocator loss at step {step}')

    loss.backward()
    gradient_norm = torch.nn.utils.clip_grad_norm_(
        head_parameters, CFG.gradient_clip
    )
    if not torch.isfinite(gradient_norm):
        raise FloatingPointError(
            f'Non-finite allocator gradient at step {step}'
        )

    allocator_optimizer.step()
    allocator_scheduler.step()
    writer.add_scalar('allocator/train_loss', loss.item(), step)

    if step == 1 or step % 25 == 0:
        record = log_metric(
            'allocator_train',
            step,
            loss=float(loss.item()),
            gradient_norm=float(gradient_norm),
            **{key: float(value) for key, value in metrics.items()},
        )
        print(record)

    if step % CFG.eval_interval == 0 or step == CFG.allocator_steps:
        model.eval()
        validation_metrics, _ = evaluate_allocator(
            model, val_dataset, SELECTED_BATCH_SIZE
        )
        model.train()
        validation_regret = validation_metrics['hybrid_regret']
        log_metric('allocator_validation', step, **validation_metrics)

        if validation_regret < best_hybrid_regret:
            best_hybrid_regret = validation_regret
            best_allocator_metrics = validation_metrics
            best_allocator_checkpoint = save_checkpoint(
                'best_allocator.pt',
                'allocator',
                model,
                allocator_optimizer,
                allocator_scheduler,
                allocator_sampler,
                step,
                validation_metrics,
            )
        sync_runtime_logs()

if best_allocator_checkpoint is None:
    raise RuntimeError('No allocator checkpoint was promoted.')

allocator_payload = torch.load(
    best_allocator_checkpoint['path'],
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(allocator_payload['model_state_dict'], strict=True)
model.eval()

test_allocator_metrics, allocation_traces = evaluate_allocator(
    model, test_dataset, SELECTED_BATCH_SIZE
)
pd.DataFrame(allocation_traces).to_parquet(
    RUN_PATHS['artifacts'] / 'allocation_traces.parquet', index=False
)

ALLOCATION_GATE = {
    'metrics': test_allocator_metrics,
    'adjoint_beats_random': (
        test_allocator_metrics['adjoint_regret']
        < test_allocator_metrics['random_regret']
    ),
    'hybrid_not_worse_than_best_component': (
        test_allocator_metrics['hybrid_regret']
        <= min(
            test_allocator_metrics['adjoint_regret'],
            test_allocator_metrics['critic_regret'],
        )
        + 1e-6
    ),
}
ALLOCATION_GATE['passed'] = bool(ALLOCATION_GATE['adjoint_beats_random'])
atomic_write_json(
    RUN_PATHS['artifacts'] / 'allocator_evaluation.json', ALLOCATION_GATE
)
print(json.dumps(ALLOCATION_GATE, indent=2))

In [ ]:
# ================================================================
# 6. Tests, system benchmarks, final report, and Drive sync
# ================================================================

TEST_RESULTS = []


def record_test(name, function):
    try:
        function()
        TEST_RESULTS.append({'name': name, 'status': 'PASS'})
    except Exception as error:
        TEST_RESULTS.append(
            {
                'name': name,
                'status': 'FAIL',
                'error': repr(error),
                'traceback': traceback.format_exc(),
            }
        )


def test_data_contract():
    assert DATA_AUDIT['status'] == 'PASS'
    assert DATA_AUDIT['real_image_payloads_verified']
    assert not DATA_AUDIT['synthetic_fallback_used']


def test_split_contract():
    assert split_sets['train'].isdisjoint(split_sets['validation'])
    assert split_sets['train'].isdisjoint(split_sets['test'])
    assert split_sets['validation'].isdisjoint(split_sets['test'])


def test_future_shift_contract():
    sample = train_dataset[0]
    assert sample['future_actions'].shape[0] == CFG.horizon
    assert sample['target_state'].shape[0] == CFG.horizon
    assert sample['context_state'].shape[0] == CFG.context_len


def test_model_finite():
    for name, tensor in model.state_dict().items():
        assert torch.isfinite(tensor).all(), name


def test_dynamics_gate():
    assert DYNAMICS_GATE['passed'], DYNAMICS_GATE


def test_allocator_negative_control():
    assert ALLOCATION_GATE['adjoint_beats_random']


def test_checkpoint_payload():
    payload = torch.load(
        best_allocator_checkpoint['path'],
        map_location='cpu',
        weights_only=False,
    )
    required = {
        'schema_version',
        'run_id',
        'stage',
        'global_step',
        'model_state_dict',
        'optimizer_state_dict',
        'scheduler_state_dict',
        'sampler_state',
        'rng_state',
        'config_hash',
        'data_manifest_hash',
        'source_hash',
        'best_metrics',
    }
    missing = required - set(payload)
    assert not missing, missing


def test_checkpoint_roundtrip():
    payload = torch.load(
        best_allocator_checkpoint['path'],
        map_location='cpu',
        weights_only=False,
    )
    clean = AdjointRecursiveWorldModel(
        state_dim=state_dim,
        action_dim=action_dim,
        visual_dim=visual_dim,
        config=CFG,
    )
    clean.load_state_dict(payload['model_state_dict'], strict=True)
    reference = {
        key: value.detach().cpu()
        for key, value in model.state_dict().items()
    }
    restored = {
        key: value.detach().cpu()
        for key, value in clean.state_dict().items()
    }
    assert reference.keys() == restored.keys()
    for key in reference:
        assert torch.equal(reference[key], restored[key]), key

record_test('data_contract', test_data_contract)
record_test('split_contract', test_split_contract)
record_test('future_shift_contract', test_future_shift_contract)
record_test('model_finite', test_model_finite)
record_test('dynamics_gate', test_dynamics_gate)
record_test('allocator_negative_control', test_allocator_negative_control)
record_test('checkpoint_payload', test_checkpoint_payload)
record_test('checkpoint_roundtrip', test_checkpoint_roundtrip)

atomic_write_json(RUN_PATHS['tests'] / 'test_results.json', TEST_RESULTS)
display(pd.DataFrame(TEST_RESULTS))
critical_failures = [
    result for result in TEST_RESULTS if result['status'] != 'PASS'
]

benchmark_loader = DataLoader(
    test_dataset,
    batch_size=min(SELECTED_BATCH_SIZE, len(test_dataset)),
    shuffle=False,
    num_workers=0,
)
benchmark_batch = prepare_batch(next(iter(benchmark_loader)))

with torch.no_grad():
    benchmark_latent = model.encode_context(
        benchmark_batch['context_visual'],
        benchmark_batch['context_state'],
        benchmark_batch['context_action'],
    )
    benchmark_effects = model.candidate_effects(benchmark_latent)


def cuda_latency_ms(function, warmup=20, iterations=100):
    for _ in range(warmup):
        function()
    torch.cuda.synchronize()
    timings = []
    for _ in range(iterations):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        function()
        end.record()
        torch.cuda.synchronize()
        timings.append(start.elapsed_time(end))
    return {
        'p50_ms': float(np.percentile(timings, 50)),
        'p95_ms': float(np.percentile(timings, 95)),
        'p99_ms': float(np.percentile(timings, 99)),
        'mean_ms': float(np.mean(timings)),
    }

with torch.no_grad():
    direct_latency = cuda_latency_ms(
        lambda: model.predict_critic(benchmark_latent, benchmark_effects)
    )
    amortized_adjoint_latency = cuda_latency_ms(
        lambda: model.predict_costate(
            benchmark_latent,
            torch.full(
                (benchmark_latent.shape[0],),
                1.0 / model.num_candidates,
                device=DEVICE,
            ),
            torch.ones(benchmark_latent.shape[0], device=DEVICE),
        )
    )

exact_adjoint_latency = cuda_latency_ms(
    lambda: exact_adjoint_targets(benchmark_batch),
    warmup=5,
    iterations=20,
)

SYSTEM_BENCHMARK = {
    'gpu': GPU_NAME,
    'gpu_vram_gib': GPU_VRAM_GIB,
    'selected_microbatch': SELECTED_BATCH_SIZE,
    'peak_training_vram_gib': float(
        valid_benchmarks['peak_vram_gib'].max()
    ),
    'direct_critic': direct_latency,
    'amortized_adjoint': amortized_adjoint_latency,
    'exact_adjoint_teacher': exact_adjoint_latency,
    'task_success': 'NOT_MEASURED',
    'task_success_reason': (
        'No external simulator or physical robot evaluation was attached '
        'to this offline dataset run.'
    ),
}
atomic_write_json(
    RUN_PATHS['benchmarks'] / 'systems_benchmark.json', SYSTEM_BENCHMARK
)

# Pilot actual-byte benchmark for quantised refinement residuals.
with torch.no_grad():
    quantized = torch.clamp(
        torch.round(benchmark_effects * 127.0), -128, 127
    ).to(torch.int8).cpu().numpy()
raw_bytes = quantized.tobytes()
compressed_bytes = zlib.compress(raw_bytes, level=9)
RATE_BENCHMARK = {
    'codec': 'zlib pilot byte count; not the final canonical coder',
    'raw_bits': len(raw_bytes) * 8,
    'compressed_bits': len(compressed_bytes) * 8,
    'bits_per_candidate_scalar': (
        len(compressed_bytes) * 8 / max(1, quantized.size)
    ),
}
atomic_write_json(
    RUN_PATHS['benchmarks'] / 'rate_benchmark.json', RATE_BENCHMARK
)

peak_vram = SYSTEM_BENCHMARK['peak_training_vram_gib']
if 'A100' in GPU_NAME:
    if peak_vram > 34.0:
        hardware_decision = (
            'TEST_A100_80GB: the measured workload is near the safe '
            '40GB operating margin.'
        )
    else:
        hardware_decision = (
            'CONTINUE_A100_40GB: the workload fits with adequate memory '
            'margin. A100 80GB is not justified.'
        )
elif 'L4' in GPU_NAME:
    if peak_vram > 20.0:
        hardware_decision = (
            'TEST_A100_40GB: the measured L4 memory margin is low.'
        )
    else:
        hardware_decision = 'CONTINUE_L4: no measured hardware bottleneck.'
else:
    hardware_decision = 'REVIEW_HARDWARE_PROFILE'

RUN_STATUS = {
    'data': 'PASS' if DATA_AUDIT['status'] == 'PASS' else 'FAIL',
    'dynamics': 'PASS' if DYNAMICS_GATE['passed'] else 'FAIL',
    'allocation': 'PASS' if ALLOCATION_GATE['passed'] else 'FAIL',
    'tests': 'PASS' if not critical_failures else 'FAIL',
    'production_status': (
        'PILOT_COMPLETE_NOT_DEPLOYMENT_READY'
        if not critical_failures
        else 'BLOCKED'
    ),
    'hardware_decision': hardware_decision,
}

REPORT = {
    'run_id': RUN_ID,
    'status': RUN_STATUS,
    'config_hash': CONFIG_HASH,
    'data_manifest_hash': DATA_MANIFEST_HASH,
    'source_hash': SOURCE_HASH,
    'best_dynamics_checkpoint': best_checkpoint,
    'best_allocator_checkpoint': best_allocator_checkpoint,
    'dynamics_evaluation': DYNAMICS_GATE,
    'allocator_evaluation': ALLOCATION_GATE,
    'systems_benchmark': SYSTEM_BENCHMARK,
    'rate_benchmark': RATE_BENCHMARK,
    'test_results': TEST_RESULTS,
}
atomic_write_json(
    RUN_PATHS['reports'] / 'acceptance_report.json', REPORT
)

summary = f"""# AdjointRWM pilot run summary

## Run

- Run ID: `{RUN_ID}`
- GPU: `{GPU_NAME}`
- Selected microbatch: `{SELECTED_BATCH_SIZE}`
- Trainable parameters: `{trainable_parameters:,}`

## Status

```json
{json.dumps(RUN_STATUS, indent=2)}
```

## Dynamics

```json
{json.dumps(DYNAMICS_GATE, indent=2)}
```

## Allocation

```json
{json.dumps(ALLOCATION_GATE, indent=2)}
```

## Hardware

{hardware_decision}

## Claim boundary

This offline run measures predictive dynamics, local refinement candidate
ranking, adjoint/critic regret, latency, memory, and a pilot rate ledger.

It does not measure physical-robot or simulator task success.
"""
atomic_write_text(
    RUN_PATHS['reports'] / 'run_summary.md', textwrap.dedent(summary)
)

writer.close()
sync_runtime_logs()

if not critical_failures:
    atomic_write_text(
        RUN_DIR / 'COMPLETE',
        datetime.datetime.now(datetime.UTC).isoformat() + '\n',
    )

print(json.dumps(REPORT, indent=2, default=str))
print('\nPersistent run directory:')
print(RUN_DIR)